# InSAR ??????(LOS)??????? (Geo ?????)

????? CDSE JupyterLab **Geo science (geo)** ?????????????????:
1. **??????**:?????????? S1A/S1D ??????;
2. **???????(Reference Point)**:?????????????(??? 0.965)???????;
3. **???????????**:?? Sentinel-1 C ????(5.55 cm)?????????????;
4. **???????**:?????????? .npy ????? .json?

In [1]:
# ?? 1:????????????
import os, json
import numpy as np

print('=== STEP 1: CONSTANTS ===')
LAMBDA_C_BAND = 0.055465
SCALE_PHASE_TO_MM = -(LAMBDA_C_BAND / (4.0 * np.pi)) * 1000.0
print(f'Radar wavelength: {LAMBDA_C_BAND * 100:.2f} cm')
print(f'Phase-to-LOS factor: {SCALE_PHASE_TO_MM:.4f} mm/rad')

=== STEP 1: CONSTANTS ===
Radar wavelength: 5.55 cm
Phase-to-LOS factor: -4.4138 mm/rad


In [2]:
# ?? 2:?? InSAR ??????
print('\n=== STEP 2: LOAD MATRICES ===')
paths = ['/home/jovyan/mystorage/everest/03_products/grd_insar_overlay', '/home/jovyan/mystorage/everest_glacier_monitor/03_products/grd_insar_overlay']
base_dir = next((p for p in paths if os.path.exists(p)), None)
print(f'Using base_dir: {base_dir}')

earlier_coh = np.load(os.path.join(base_dir, 'insar_coherence_earlier_common_grid.npy'))
latest_coh = np.load(os.path.join(base_dir, 'insar_coherence_latest_common_grid.npy'))
delta_coh = np.load(os.path.join(base_dir, 'insar_delta_coherence_common_grid.npy'))
combined_mask = np.load(os.path.join(base_dir, 'grd_insar_combined_candidate_mask.npy'))

print(f'Grid shape: {earlier_coh.shape}')
print(f'Earlier coh mean: {np.nanmean(earlier_coh):.3f}')
print(f'Latest coh mean: {np.nanmean(latest_coh):.3f}')

=== STEP 2: LOAD MATRICES ===
Using base_dir: /home/jovyan/mystorage/everest/03_products/grd_insar_overlay
Grid shape: (686, 1000)
Earlier coh mean: 0.486
Latest coh mean: 0.429


In [3]:
# ?? 3:??????????????
print('\n=== STEP 3: ANCHOR SELECTION & DISPLACEMENT ===')
stability_score = (earlier_coh + latest_coh) - 2.0 * np.abs(delta_coh)
stability_score[np.isnan(stability_score)] = -999.0

best_idx = np.unravel_index(np.argmax(stability_score), stability_score.shape)
anchor_y, anchor_x = int(best_idx[0]), int(best_idx[1])
anchor_coh = float(latest_coh[anchor_y, anchor_x])
print(f'Optimal bedrock anchor at (Y={anchor_y}, X={anchor_x}), Coherence={anchor_coh:.3f}')

relative_delta = delta_coh - delta_coh[anchor_y, anchor_x]
disp_los_mm = relative_delta * SCALE_PHASE_TO_MM
disp_los_mm_candidates = np.where(combined_mask, disp_los_mm, np.nan)
valid_disp = disp_los_mm_candidates[~np.isnan(disp_los_mm_candidates)]

print(f'Evaluated candidate pixels: {len(valid_disp)}')
print(f'Displacement range: [{np.min(valid_disp):.2f} mm, {np.max(valid_disp):.2f} mm]')
print(f'Median displacement: {np.median(valid_disp):.2f} mm')
print(f'Mean displacement: {np.mean(valid_disp):.2f} mm')

=== STEP 3: ANCHOR SELECTION & DISPLACEMENT ===
Optimal bedrock anchor at (Y=285, X=613), Coherence=0.965
Evaluated candidate pixels: 20278
Displacement range: [0.46 mm, 2.75 mm]
Median displacement: 0.66 mm
Mean displacement: 0.79 mm


In [4]:
# ?? 4:??????
print('\n=== STEP 4: SAVE OUTPUTS ===')
out_dirs = ['/home/jovyan/mystorage/everest/03_products/insar', '/home/jovyan/mystorage/everest_glacier_monitor/03_products/insar']
for od in out_dirs:
    os.makedirs(od, exist_ok=True)
    np.save(os.path.join(od, 'everest_insar_los_displacement_candidates_mm.npy'), disp_los_mm_candidates)
    summary = {'product': 'Everest InSAR LOS Displacement Screening', 'evaluated_pixels': int(len(valid_disp)), 'min_mm': float(np.min(valid_disp)), 'max_mm': float(np.max(valid_disp)), 'median_mm': float(np.median(valid_disp)), 'mean_mm': float(np.mean(valid_disp))}
    with open(os.path.join(od, 'everest_insar_displacement_summary.json'), 'w') as f: json.dump(summary, f, indent=2)
    print('Saved to:', os.path.join(od, 'everest_insar_displacement_summary.json'))

=== STEP 4: SAVE OUTPUTS ===
Saved to: /home/jovyan/mystorage/everest/03_products/insar/everest_insar_displacement_summary.json
Saved to: /home/jovyan/mystorage/everest_glacier_monitor/03_products/insar/everest_insar_displacement_summary.json
